In [1]:
# Bibliothèques de base
import pyodbc
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns


C:\Users\ayadi\anaconda3\anaconda\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# Connexion à la base de données
conn_str = (
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=LAPTOP-GPVK1ANC;"  # Remplacez par votre serveur
    "Database=DW_SupplyChain;"
    "Trusted_Connection=yes;"
)
cnxn = pyodbc.connect(conn_str)

# Requête SQL pour extraire les données nécessaires
query = """
SELECT 
    productFK,
    brandFK,
    Production_Start_Time,
    Production_End_Time,
    manufacturedate,
    expirydate,
    Dosage,
    quantityused
FROM fact_production
"""
# Chargement des données dans un DataFrame
df = pd.read_sql(query, cnxn)

# Vérification des premières lignes du DataFrame
df.head()


C:\Users\ayadi\AppData\Local\Temp\ipykernel_18772\1517602925.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, cnxn)


,productFK,brandFK,Production_Start_Time,Production_End_Time,manufacturedate,expirydate,Dosage,quantityused
0,2,15,2022032723,2022032800,2024082023,2025083023,9,9
1,2,15,2022040823,2022040806,2024082023,2025083023,9,9
2,2,15,2022050523,2022050517,2024082023,2025083023,9,9
3,2,15,2022052123,2022052119,2024082023,2025083023,9,9
4,2,15,2022070123,2022070113,2024082023,2025083023,9,9


In [3]:
# Conversion des colonnes de dates en datetime
df['Production_Start_Time'] = pd.to_datetime(df['Production_Start_Time'], format='%Y%m%d%H', errors='coerce')
df['Production_End_Time'] = pd.to_datetime(df['Production_End_Time'], format='%Y%m%d%H', errors='coerce')
df['manufacturedate'] = pd.to_datetime(df['manufacturedate'], format='%Y%m%d%H', errors='coerce')
df['expirydate'] = pd.to_datetime(df['expirydate'], format='%Y%m%d%H', errors='coerce')

# Vérification du résultat
df[['Production_Start_Time', 'Production_End_Time', 'manufacturedate', 'expirydate']].head()


,Production_Start_Time,Production_End_Time,manufacturedate,expirydate
0,2022-03-27 23:00:00,2022-03-28 00:00:00,2024-08-20 23:00:00,2025-08-30 23:00:00
1,2022-04-08 23:00:00,2022-04-08 06:00:00,2024-08-20 23:00:00,2025-08-30 23:00:00
2,2022-05-05 23:00:00,2022-05-05 17:00:00,2024-08-20 23:00:00,2025-08-30 23:00:00
3,2022-05-21 23:00:00,2022-05-21 19:00:00,2024-08-20 23:00:00,2025-08-30 23:00:00
4,2022-07-01 23:00:00,2022-07-01 13:00:00,2024-08-20 23:00:00,2025-08-30 23:00:00


In [4]:
# Calcul conditionnel de la durée de production
df['production_duration_hours'] = np.where(
    df['Production_End_Time'] >= df['Production_Start_Time'],
    (df['Production_End_Time'] - df['Production_Start_Time']).dt.total_seconds() / 3600,
    (df['Production_Start_Time'] - df['Production_End_Time']).dt.total_seconds() / 3600
)


In [5]:
# Vérification
df[['Production_Start_Time', 'Production_End_Time', 'production_duration_hours']].head()


,Production_Start_Time,Production_End_Time,production_duration_hours
0,2022-03-27 23:00:00,2022-03-28 00:00:00,1.0
1,2022-04-08 23:00:00,2022-04-08 06:00:00,17.0
2,2022-05-05 23:00:00,2022-05-05 17:00:00,6.0
3,2022-05-21 23:00:00,2022-05-21 19:00:00,4.0
4,2022-07-01 23:00:00,2022-07-01 13:00:00,10.0


In [6]:
# Sélection des features et de la cible
features = ['productFK', 'brandFK', 'Dosage', 'production_duration_hours']
target = 'quantityused'

X = df[features]
y = df[target]

# Vérification de la dimension des données
X.shape, y.shape


((33043, 4), (33043,))

In [7]:
# Normalisation des features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [8]:
from sklearn.linear_model import Ridge
from sklearn.svm import SVR

# Séparation des données
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Initialisation des modèles
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Support Vector Regression": SVR()
}

# Dictionnaire pour stocker les résultats
results = {}

# Entraînement et évaluation
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    }

# Affichage des résultats
pd.DataFrame(results).T


,MSE,RMSE,R2 Score
Linear Regression,0.343581,0.586157,0.959142
Ridge Regression,0.343578,0.586155,0.959143
Support Vector Regression,0.343252,0.585877,0.959182


In [11]:
# Initialisation du dictionnaire pour stocker les prédictions
predictions_table = pd.DataFrame({'Actualquantityused': y_test.reset_index(drop=True)})

# Entraînement et évaluation
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Ajout des prédictions dans le tableau
    predictions_table[name] = y_pred

    # Calcul des métriques
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results[name] = {
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    }

# Affichage des métriques
print("🔍 Résultats des modèles :")
display(pd.DataFrame(results).T)

# Affichage du tableau comparatif des prédictions
print("\n📊 Comparaison des prédictions :")
display(predictions_table.head(10))  # Modifier le nombre de lignes affichées si besoin


🔍 Résultats des modèles :


,MSE,RMSE,R2 Score
Linear Regression,0.343581,0.586157,0.959142
Ridge Regression,0.343578,0.586155,0.959143
Support Vector Regression,0.343252,0.585877,0.959182



📊 Comparaison des prédictions :


,Actualquantityused,Linear Regression,Ridge Regression,Support Vector Regression
0,5,4.999860,4.999877,4.949825
1,9,8.931906,8.931775,9.001836
2,5,5.003195,5.003195,4.986941
3,3,3.024562,3.024654,2.941213
4,8,7.945668,7.945574,7.933829
5,6,5.975529,5.975508,5.936405
6,1,1.051458,1.051624,1.050164
7,2,2.050239,2.050368,1.940568
8,8,7.960374,7.960280,7.997391
9,5,4.995977,4.995994,4.939777
